# PSE Stock Model Training (Run Once)

This notebook trains two distinct models on the PSE OHLCV dataset and saves artifacts for the Flask playback app:

- A forecast model that predicts the next close for playback and ghost forecasting.
- A trading-policy model that predicts BUY / SELL / HOLD actions to simulate portfolio growth from a starting cash amount.

Both models use the same chronological train/test split, but they solve different problems and are exported separately.

In [1]:
from pathlib import Path
import json

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
 )
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

In [2]:
BASE_DIR = Path.cwd()
DATA_PATH = BASE_DIR / "data" / "pse_multistock_ohlcv.csv"
ARTIFACT_DIR = BASE_DIR / "graphanimation" / "artifacts"

FORECAST_MODEL_PATH = ARTIFACT_DIR / "forecast_model.joblib"
FORECAST_PREDICTIONS_PATH = ARTIFACT_DIR / "forecast_predictions.csv"
FORECAST_METRICS_PATH = ARTIFACT_DIR / "forecast_metrics.json"

TRADING_MODEL_PATH = ARTIFACT_DIR / "trading_model.joblib"
TRADING_BACKTEST_PATH = ARTIFACT_DIR / "trading_backtest.csv"
TRADING_METRICS_PATH = ARTIFACT_DIR / "trading_metrics.json"

METRICS_PATH = ARTIFACT_DIR / "metrics.json"

TEST_START_DATE = pd.Timestamp("2022-03-01")
FORECAST_LAGS = 20
VOLUME_LAGS = 10
TRADE_HORIZON = 5
STARTING_CASH_REFERENCE = 100_000.0
BUY_THRESHOLD = 0.012
SELL_THRESHOLD = 0.012

assert DATA_PATH.exists(), f"Missing dataset: {DATA_PATH}"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Dataset:", DATA_PATH)
print("Artifacts:", ARTIFACT_DIR)
print("Test start date:", TEST_START_DATE.date())
print("Forecast lags:", FORECAST_LAGS)
print("Trading horizon:", TRADE_HORIZON)

Dataset: /Users/mac/Programming/FinanceBot2/data/pse_multistock_ohlcv.csv
Artifacts: /Users/mac/Programming/FinanceBot2/graphanimation/artifacts
Test start date: 2022-03-01
Forecast lags: 20
Trading horizon: 5


In [3]:
def build_features(
    df: pd.DataFrame,
    close_lags: int = FORECAST_LAGS,
    volume_lags: int = VOLUME_LAGS,
    return_windows: tuple[int, ...] = (1, 3, 5, 10),
    close_sma_windows: tuple[int, ...] = (5, 10, 20),
    volume_sma_windows: tuple[int, ...] = (3, 10),
) -> pd.DataFrame:
    work = df.copy()
    work["date"] = pd.to_datetime(work["date"])
    work = work.sort_values(["symbol", "date"]).reset_index(drop=True)

    for lag in range(1, close_lags + 1):
        work[f"lag_close_{lag}"] = work.groupby("symbol")["close"].shift(lag)
    for lag in range(1, volume_lags + 1):
        work[f"lag_volume_{lag}"] = work.groupby("symbol")["volume"].shift(lag)

    for window in return_windows:
        work[f"return_{window}"] = work[f"lag_close_1"] / work[f"lag_close_{window + 1}"] - 1.0

    for window in close_sma_windows:
        close_lag_cols = [f"lag_close_{i}" for i in range(1, window + 1)]
        work[f"sma_close_{window}"] = work[close_lag_cols].mean(axis=1)
        work[f"close_vs_sma_{window}"] = work[f"lag_close_1"] / work[f"sma_close_{window}"] - 1.0

    for window in volume_sma_windows:
        volume_lag_cols = [f"lag_volume_{i}" for i in range(1, window + 1)]
        work[f"sma_volume_{window}"] = work[volume_lag_cols].mean(axis=1)
        work[f"volume_vs_sma_{window}"] = work[f"lag_volume_1"] / work[f"sma_volume_{window}"] - 1.0

    work = work.rename(columns={"close": "target_close"})

    keep_cols = [
        "date",
        "symbol",
        "target_close",
    ] + [
        column
        for column in work.columns
        if column.startswith("lag_close_")
        or column.startswith("lag_volume_")
        or column.startswith("return_")
        or column.startswith("sma_close_")
        or column.startswith("close_vs_sma_")
        or column.startswith("sma_volume_")
        or column.startswith("volume_vs_sma_")
    ]

    return work[keep_cols].dropna().reset_index(drop=True)


def split_train_test_by_date(model_df: pd.DataFrame, test_start_date: pd.Timestamp):
    model_df = model_df.sort_values(["symbol", "date"]).reset_index(drop=True)
    train_df = model_df[model_df["date"] < test_start_date].copy()
    test_df = model_df[model_df["date"] >= test_start_date].copy()

    if train_df.empty or test_df.empty:
        raise ValueError("Train/test split is empty. Adjust test_start_date.")

    if train_df["date"].max() >= test_df["date"].min():
        raise ValueError("Date leakage detected between train and test sets.")

    return train_df, test_df


def build_trading_targets(model_df: pd.DataFrame, horizon: int = TRADE_HORIZON) -> pd.DataFrame:
    work = model_df.sort_values(["symbol", "date"]).reset_index(drop=True).copy()
    work["future_close"] = work.groupby("symbol")["target_close"].shift(-horizon)
    work["future_return"] = work["future_close"] / work["target_close"] - 1.0

    work = work.dropna(subset=["future_close"]).copy()

    work["action_label"] = np.select(
        [
            work["future_return"] > BUY_THRESHOLD,
            work["future_return"] < -SELL_THRESHOLD,
        ],
        ["BUY", "SELL"],
        default="HOLD",
    )

    return work


def simulate_trading_path(
    rows: pd.DataFrame,
    predicted_actions: np.ndarray,
    starting_cash: float = STARTING_CASH_REFERENCE,
) -> pd.DataFrame:
    cash = float(starting_cash)
    shares = 0.0
    records: list[dict[str, float | str | int]] = []

    if rows.empty:
        return pd.DataFrame()

    first_price = float(rows["target_close"].iloc[0])
    buy_hold_shares = starting_cash / first_price if first_price > 0 else 0.0

    for row_index, (_, row) in enumerate(rows.iterrows()):
        price = float(row["target_close"])
        action = str(predicted_actions[row_index])
        if action == "BUY" and cash > 0 and price > 0:
            shares += cash / price
            cash = 0.0
        elif action == "SELL" and shares > 0:
            cash += shares * price
            shares = 0.0

        portfolio_value = cash + shares * price
        buy_hold_value = buy_hold_shares * price
        records.append(
            {
                "date": row["date"],
                "symbol": row["symbol"],
                "target_close": price,
                "action": action,
                "cash": float(cash),
                "shares": float(shares),
                "portfolio_value": float(portfolio_value),
                "buy_hold_value": float(buy_hold_value),
            }
        )

    return pd.DataFrame(records)

In [4]:
raw_df = pd.read_csv(DATA_PATH)
model_df = build_features(raw_df)

numeric_features = [c for c in model_df.columns if c not in {"date", "symbol", "target_close"}]
categorical_features = ["symbol"]

forecast_train_df, forecast_test_df = split_train_test_by_date(model_df, test_start_date=TEST_START_DATE)

forecast_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

forecast_model = Pipeline(
    steps=[
        ("preprocessor", forecast_preprocessor),
        (
            "regressor",
            RandomForestRegressor(
                n_estimators=700,
                random_state=42,
                n_jobs=-1,
                max_depth=18,
                min_samples_leaf=1,
                min_samples_split=4,
            ),
        ),
    ]
)

x_train = forecast_train_df[categorical_features + numeric_features]
y_train = forecast_train_df["target_close"]
x_test = forecast_test_df[categorical_features + numeric_features]
y_test = forecast_test_df["target_close"]

forecast_model.fit(x_train, y_train)
forecast_pred = forecast_model.predict(x_test)

forecast_naive_pred = x_test["lag_close_1"].to_numpy()
forecast_rmse = float(np.sqrt(mean_squared_error(y_test, forecast_pred)))
forecast_naive_rmse = float(np.sqrt(mean_squared_error(y_test, forecast_naive_pred)))

forecast_metrics = {
    "overall": {
        "rmse": forecast_rmse,
        "mae": float(mean_absolute_error(y_test, forecast_pred)),
        "r2": float(r2_score(y_test, forecast_pred)),
        "naive_rmse": forecast_naive_rmse,
        "naive_mae": float(mean_absolute_error(y_test, forecast_naive_pred)),
        "rmse_improvement_vs_naive_pct": float((forecast_naive_rmse - forecast_rmse) / forecast_naive_rmse * 100.0) if forecast_naive_rmse > 0 else 0.0,
        "n_train": int(len(forecast_train_df)),
        "n_test": int(len(forecast_test_df)),
        "n_numeric_features": int(len(numeric_features)),
        "test_start_date": str(TEST_START_DATE.date()),
        "actual_first_test_date": str(forecast_test_df["date"].min().date()),
    },
    "feature_config": {
        "close_lags": FORECAST_LAGS,
        "volume_lags": VOLUME_LAGS,
        "return_windows": [1, 3, 5, 10],
        "close_sma_windows": [5, 10, 20],
        "volume_sma_windows": [3, 10],
    },
    "model_config": {
        "type": "RandomForestRegressor",
        "n_estimators": 700,
        "max_depth": 18,
        "min_samples_leaf": 1,
        "min_samples_split": 4,
    },
    "by_symbol": {},
}

forecast_predictions_df = forecast_test_df[["date", "symbol", "target_close"]].copy()
forecast_predictions_df["pred_close"] = forecast_pred

for symbol, g in forecast_predictions_df.groupby("symbol"):
    actual = g["target_close"].to_numpy()
    pred = g["pred_close"].to_numpy()
    forecast_metrics["by_symbol"][symbol] = {
        "rmse": float(np.sqrt(mean_squared_error(actual, pred))),
        "mae": float(mean_absolute_error(actual, pred)),
        "r2": float(r2_score(actual, pred)),
        "n_test": int(len(g)),
    }

trading_df = build_trading_targets(model_df, horizon=TRADE_HORIZON)
trading_train_cutoff = TEST_START_DATE - pd.offsets.BDay(TRADE_HORIZON)
trading_train_df = trading_df[trading_df["date"] < trading_train_cutoff].copy()
trading_test_df = trading_df[trading_df["date"] >= TEST_START_DATE].copy()

trading_preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            Pipeline(steps=[("imputer", SimpleImputer(strategy="median"))]),
            numeric_features,
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

trading_model = Pipeline(
    steps=[
        ("preprocessor", trading_preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=700,
                random_state=42,
                n_jobs=-1,
                max_depth=18,
                min_samples_leaf=2,
                min_samples_split=4,
                class_weight="balanced_subsample",
            ),
        ),
    ]
)

trading_x_train = trading_train_df[categorical_features + numeric_features]
trading_y_train = trading_train_df["action_label"]
trading_x_test = trading_test_df[categorical_features + numeric_features]
trading_y_test = trading_test_df["action_label"]

trading_model.fit(trading_x_train, trading_y_train)
trading_pred = trading_model.predict(trading_x_test)

trading_metrics = {
    "overall": {
        "accuracy": float(accuracy_score(trading_y_test, trading_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(trading_y_test, trading_pred)),
        "macro_f1": float(f1_score(trading_y_test, trading_pred, average="macro")),
        "macro_precision": float(precision_score(trading_y_test, trading_pred, average="macro", zero_division=0)),
        "macro_recall": float(recall_score(trading_y_test, trading_pred, average="macro", zero_division=0)),
        "n_train": int(len(trading_train_df)),
        "n_test": int(len(trading_test_df)),
        "label_counts_train": {label: int(count) for label, count in trading_y_train.value_counts().items()},
        "label_counts_test": {label: int(count) for label, count in trading_y_test.value_counts().items()},
        "test_start_date": str(TEST_START_DATE.date()),
    },
    "feature_config": {
        "close_lags": FORECAST_LAGS,
        "volume_lags": VOLUME_LAGS,
        "return_windows": [1, 3, 5, 10],
        "close_sma_windows": [5, 10, 20],
        "volume_sma_windows": [3, 10],
        "label_horizon": TRADE_HORIZON,
        "buy_threshold": BUY_THRESHOLD,
        "sell_threshold": SELL_THRESHOLD,
    },
    "model_config": {
        "type": "RandomForestClassifier",
        "n_estimators": 700,
        "max_depth": 18,
        "min_samples_leaf": 2,
        "min_samples_split": 4,
        "class_weight": "balanced_subsample",
    },
    "by_symbol": {},
}

trading_backtest_frames: list[pd.DataFrame] = []

for symbol, symbol_rows in trading_test_df.groupby("symbol"):
    symbol_rows = symbol_rows.sort_values("date").reset_index(drop=True)
    symbol_x = symbol_rows[categorical_features + numeric_features]
    symbol_pred = trading_model.predict(symbol_x)
    symbol_sim = simulate_trading_path(symbol_rows, symbol_pred, STARTING_CASH_REFERENCE)
    if symbol_sim.empty:
        continue

    symbol_sim["predicted_action"] = symbol_pred
    trading_backtest_frames.append(symbol_sim)

    final_value = float(symbol_sim["portfolio_value"].iloc[-1])
    buy_hold_final_value = float(symbol_sim["buy_hold_value"].iloc[-1])
    trading_metrics["by_symbol"][symbol] = {
        "final_value": final_value,
        "final_return_pct": float((final_value / STARTING_CASH_REFERENCE - 1.0) * 100.0),
        "buy_hold_final_value": buy_hold_final_value,
        "buy_hold_return_pct": float((buy_hold_final_value / STARTING_CASH_REFERENCE - 1.0) * 100.0),
        "n_test": int(len(symbol_sim)),
        "action_counts": {label: int(count) for label, count in pd.Series(symbol_pred).value_counts().items()},
    }

if trading_backtest_frames:
    trading_backtest_df = pd.concat(trading_backtest_frames, ignore_index=True)
else:
    trading_backtest_df = pd.DataFrame(columns=["date", "symbol", "target_close", "action", "cash", "shares", "portfolio_value", "buy_hold_value", "predicted_action"])

trade_returns = [entry["final_return_pct"] for entry in trading_metrics["by_symbol"].values()]
buy_hold_returns = [entry["buy_hold_return_pct"] for entry in trading_metrics["by_symbol"].values()]
trading_metrics["overall"].update(
    {
        "mean_final_return_pct": float(np.mean(trade_returns)) if trade_returns else 0.0,
        "mean_buy_hold_return_pct": float(np.mean(buy_hold_returns)) if buy_hold_returns else 0.0,
        "mean_return_advantage_pct": float(np.mean(np.array(trade_returns) - np.array(buy_hold_returns))) if trade_returns else 0.0,
        "n_symbols": int(len(trading_metrics["by_symbol"])),
    }
)

print("Forecast train max date:", forecast_train_df["date"].max().date())
print("Forecast test min date:", forecast_test_df["date"].min().date())
print("Forecast numeric features:", len(numeric_features))
print("Trading train max date:", trading_train_df["date"].max().date())
print("Trading test min date:", trading_test_df["date"].min().date())
print("Trading labels:", trading_metrics["overall"]["label_counts_train"])
forecast_metrics["overall"]
trading_metrics["overall"]

Forecast train max date: 2022-02-28
Forecast test min date: 2022-03-01
Forecast numeric features: 44
Trading train max date: 2022-02-21
Trading test min date: 2022-03-01
Trading labels: {'BUY': 15520, 'SELL': 13833, 'HOLD': 11460}


{'accuracy': 0.3596341340838275,
 'balanced_accuracy': 0.36593688809935654,
 'macro_f1': 0.35765919829709497,
 'macro_precision': 0.3660311193747718,
 'macro_recall': 0.36593688809935654,
 'n_train': 40813,
 'n_test': 9949,
 'label_counts_train': {'BUY': 15520, 'SELL': 13833, 'HOLD': 11460},
 'label_counts_test': {'SELL': 3765, 'BUY': 3370, 'HOLD': 2814},
 'test_start_date': '2022-03-01',
 'mean_final_return_pct': 15.162970876832315,
 'mean_buy_hold_return_pct': -24.468988626116513,
 'mean_return_advantage_pct': 39.63195950294883,
 'n_symbols': 10}

In [5]:
forecast_model_path = FORECAST_MODEL_PATH
forecast_predictions_path = FORECAST_PREDICTIONS_PATH
forecast_metrics_path = FORECAST_METRICS_PATH

trading_model_path = TRADING_MODEL_PATH
trading_backtest_path = TRADING_BACKTEST_PATH
trading_metrics_path = TRADING_METRICS_PATH

forecast_model_path.parent.mkdir(parents=True, exist_ok=True)

joblib.dump(forecast_model, forecast_model_path)
forecast_predictions_df.to_csv(forecast_predictions_path, index=False)
forecast_metrics_path.write_text(json.dumps(forecast_metrics, indent=2), encoding="utf-8")

joblib.dump(trading_model, trading_model_path)
trading_backtest_df.to_csv(trading_backtest_path, index=False)
trading_metrics_path.write_text(json.dumps(trading_metrics, indent=2), encoding="utf-8")

combined_metrics = {
    "forecast": forecast_metrics,
    "trading": trading_metrics,
    "defaults": {
        "default_model": "trading",
        "forecast_horizon": TRADE_HORIZON,
        "starting_cash_reference": STARTING_CASH_REFERENCE,
    },
}
METRICS_PATH.write_text(json.dumps(combined_metrics, indent=2), encoding="utf-8")

print("Saved:")
print(" -", forecast_model_path)
print(" -", forecast_predictions_path)
print(" -", forecast_metrics_path)
print(" -", trading_model_path)
print(" -", trading_backtest_path)
print(" -", trading_metrics_path)
print(" -", METRICS_PATH)

Saved:
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/forecast_model.joblib
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/forecast_predictions.csv
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/forecast_metrics.json
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/trading_model.joblib
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/trading_backtest.csv
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/trading_metrics.json
 - /Users/mac/Programming/FinanceBot2/graphanimation/artifacts/metrics.json


## Run Instructions

1. Select the project `.venv` kernel for this notebook.
2. Run all cells from top to bottom once.
3. The notebook exports separate forecast and trading artifacts into `graphanimation/artifacts/`.
4. Start the web app with:

```bash
/Users/mac/Programming/FinanceBot2/.venv/bin/python graphanimation/app.py
```

The Flask app will read both model artifacts from the artifacts folder and switch behavior based on the selected model.